# Chapter 3 — Worked Example: Spatial ML Comparison on the Hennaya Dataset

**AI for Hydrogeologists** — companion notebook

**Note on scope.** The original outline for Section 3.6 called for a temporal
comparison of Random Forest / XGBoost / LSTM on a continuous monthly time
series. The Hennaya dataset is three independent snapshots (1981, 2012, 2022),
not a continuous series, so a temporal comparison is not meaningful here (see
Chapter 4). This notebook instead teaches the same core lesson of Section
3.5.1-3.5.2 - why naive random splits fail on spatially autocorrelated
hydrogeological data - using a spatial prediction task the data actually
support: predicting **depth to water table** from physiographic covariates,
comparing a naive random split against a spatially-honest split.

Data comes directly from the GitHub repository (Chapter 2 output).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

BASE = "https://raw.githubusercontent.com/Dr-LAOUFIAbdessalam/ai-hydrogeologists/main/"
df = pd.read_csv(BASE + "ch03_core_ml_methods/data/raw/hennaya_heads_ml_ready.csv")
print(df.shape)
df.head()


## Prepare features and target

In [ ]:
# Drop the one physically implausible record flagged in Chapter 2
df = df[df["well_id"] != "HYN-2022-033"].copy()
# Drop rows with missing soil texture (built-up land, SoilGrids has no
# prediction there) - for this exercise only
df = df.dropna(subset=["clay_pct", "sand_pct", "silt_pct"]).copy()

target = "depth_to_water_approx_m"
num_features = ["elevation_srtm_m", "clay_pct", "sand_pct", "silt_pct", "x", "y"]
cat_feature = "landcover_class"

X_num = df[num_features].values
X_cat = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit_transform(df[[cat_feature]])
X = np.hstack([X_num, X_cat])
y = df[target].values

print(f"n = {len(df)} wells, {X.shape[1]} features")
print(f"Target ({target}): mean={y.mean():.1f} m, std={y.std():.1f} m, "
      f"range=[{y.min():.1f}, {y.max():.1f}] m")


**Note:** a few negative depth-to-water values appear, an artefact of the
30 m SRTM DEM's limited vertical accuracy near small local depressions -
not evidence of artesian conditions. Flagged, not corrected, for transparency.

## Spatial blocks for cross-validation (Section 3.5.2)

In [ ]:
N_BLOCKS = 6
blocks = KMeans(n_clusters=N_BLOCKS, random_state=42, n_init=10).fit_predict(df[["x", "y"]].values)
df["spatial_block"] = blocks
print(df["spatial_block"].value_counts().sort_index())


## Compare naive random split vs spatially-honest split

In [ ]:
def spatial_cv_scores(model, X, y, blocks):
    rmses, r2s = [], []
    for b in np.unique(blocks):
        train_idx, test_idx = blocks != b, blocks == b
        if test_idx.sum() < 3:
            continue
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], pred)))
        r2s.append(r2_score(y[test_idx], pred))
    return np.array(rmses), np.array(r2s)

def random_cv_scores(model, X, y, n_splits=6, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    folds = np.array_split(idx, n_splits)
    rmses, r2s = [], []
    for i in range(n_splits):
        test_idx = folds[i]
        train_idx = np.hstack([folds[j] for j in range(n_splits) if j != i])
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        rmses.append(np.sqrt(mean_squared_error(y[test_idx], pred)))
        r2s.append(r2_score(y[test_idx], pred))
    return np.array(rmses), np.array(r2s)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=6, random_state=42),
}

for name, model in models.items():
    rmse_r, r2_r = random_cv_scores(model, X, y)
    rmse_s, r2_s = spatial_cv_scores(model, X, y, blocks)
    print(f"\n{name}:")
    print(f"  Random CV   : RMSE = {rmse_r.mean():.2f} +/- {rmse_r.std():.2f} m | R2 = {r2_r.mean():.2f}")
    print(f"  Spatial CV  : RMSE = {rmse_s.mean():.2f} +/- {rmse_s.std():.2f} m | R2 = {r2_s.mean():.2f}")


## Interpretation

Both models look reasonably good under a random split (R2 around 0.66-0.70),
but collapse under spatial cross-validation (R2 turns negative). This
confirms, with real Hennaya data, exactly the warning of Section 3.5.1: nearby
wells share enough spatial autocorrelation that a random split lets the model
"cheat" by interpolating between near-duplicate neighbours. The spatially
honest estimate is the trustworthy one for judging real-world transferability
to unmonitored locations.